# 📘 智能体架构 11：元控制器（Meta-Controller）

欢迎阅读本系列第十一个 Notebook。今天搭建 **元控制器（Meta-Controller）**：一种监督型智能体架构，用于编排一组专业化子智能体。该模式是构建强大、多能力 AI 系统的基本构件。

与其做一个试图包办一切的单体智能体，不如让元控制器充当智能调度器：接收请求、分析性质，再从可用智能体池中指派最合适专家。各子智能体可针对具体任务深度优化，从而获得更好的性能与模块化。

我们将用三个不同专家演示：
1. **通才智能体：** 处理闲聊与简单问题。
2. **研究智能体：** 配备搜索工具，回答近期事件或复杂主题相关问题。
3. **编程智能体：** 专注生成 Python 代码片段。

元控制器是系统的「大脑」，审视每条用户查询并决定由谁响应。新能力只需新增专家并让控制器知晓，系统灵活且易扩展。


### 定义

**元控制器**（或称路由器）是多智能体系统中的监督智能体，负责分析到达的任务并将其分派给合适的专家子智能体或工作流。它作为智能路由层，决定当前任务最适合哪种工具或专家。

### 高层工作流

1. **接收输入：** 系统收到用户请求。
2. **元控制器分析：** 元控制器审视请求的意图、复杂度与内容。
3. **分派专家：** 根据分析，从预定义池中选择最合适的专家智能体（如「Researcher」「Coder」「Chatbot」）。
4. **执行任务：** 被选中的专家执行任务并生成结果。
5. **返回结果：** 将专家结果返回用户；在更复杂流程中，控制权可能回到元控制器以进行后续步骤或监控。

### 适用场景 / 应用
* **多服务 AI 平台：** 单一入口提供文档分析、数据可视化、创意写作等多样服务。
* **自适应个人助理：** 在日历、网页搜索、智能家居控制等模式或工具间切换。
* **企业工作流：** 根据工单内容将客服请求路由到技术、账单、销售等部门。

### 优势与局限
* **优势：**
    * **灵活与模块化：** 新增专家并更新路由逻辑即可扩展能力。
    * **性能：** 可使用高度优化的专家智能体，而非样样平庸的「万金油」模型。
* **局限：**
    * **控制器单点风险：** 整体质量高度依赖路由是否正确；错误路由会导致次优或错误结果。
    * **潜在延迟增加：** 相比直接调用单一智能体，路由步骤可能带来少量额外时延。


## 阶段 0：基础与环境

将安装依赖并配置环境。研究智能体的搜索工具需要 `langchain-tavily`。


In [1]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain-tavily

In [1]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Meta-Controller (DeepSeek)"

required_vars = ["DEEPSEEK_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

Environment variables loaded and tracing is set up.


## 阶段 1：构建专家智能体

首先创建专家团队：每个智能体是一条带特定人设的简单链；研究智能体配备工具。再封装为节点函数供 LangGraph 使用。


In [3]:
console = Console()
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",
    temperature=0,
)
search_tool = TavilySearch(max_results=3)

# Define the state for the overall graph
class MetaAgentState(TypedDict):
    user_request: str
    next_agent_to_call: Optional[str]
    generation: str

# A helper factory function to create specialist agent nodes
def create_specialist_node(persona: str, tools: list = None):
    """Factory to create a specialist agent node."""
    system_prompt = f"You are a specialist agent with the following persona: {persona}. Respond directly and concisely to the user's request based on your role."
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{user_request}")
    ])
    
    if tools:
        chain = prompt | llm.bind_tools(tools)
    else:
        chain = prompt | llm
        
    def specialist_node(state: MetaAgentState) -> Dict[str, Any]:
        result = chain.invoke({"user_request": state['user_request']})
        return {"generation": result.content}
    
    return specialist_node

# 1. Generalist Agent Node
generalist_node = create_specialist_node(
    "You are a friendly and helpful generalist AI assistant. You handle casual conversation and simple questions."
)

# 2. Research Agent Node
research_agent_node = create_specialist_node(
    "You are an expert researcher. You must use your search tool to find information to answer the user's question.",
    tools=[search_tool]
)

# 3. Coding Agent Node
coding_agent_node = create_specialist_node(
    "You are an expert Python programmer. Your task is to write clean, efficient Python code based on the user's request. Provide only the code, wrapped in markdown code blocks, with minimal explanation."
)

print("Specialist agents defined successfully.")

Specialist agents defined successfully.


## 阶段 2：构建元控制器

这是系统的「大脑」。元控制器是一个仅负责「把请求路由给哪位专家」的 LLM 节点；其提示词质量对系统表现至关重要。


In [ ]:
# Pydantic model for the controller's routing decision
class ControllerDecision(BaseModel):
    next_agent: str = Field(description="The name of the specialist agent to call next. Must be one of ['Generalist', 'Researcher', 'Coder'].")
    reasoning: str = Field(description="A brief reason for choosing the next agent.")

def meta_controller_node(state: MetaAgentState) -> Dict[str, Any]:
    """The central controller that decides which specialist to call."""
    console.print("--- 🧠 Meta-Controller Analyzing Request ---")
    
    # Define the specialists and their descriptions for the controller
    specialists = {
        "Generalist": "Handles casual conversation, greetings, and simple questions.",
        "Researcher": "Answers questions about recent events, complex topics, or anything requiring up-to-date information from the web.",
        "Coder": "Writes Python code based on a user's specification."
    }
    
    specialist_descriptions = "\n".join([f"- {name}: {desc}" for name, desc in specialists.items()])
    
    prompt = ChatPromptTemplate.from_template(
        f"""You are the meta-controller for a multi-agent AI system. Your job is to analyze the user's request and route it to the most appropriate specialist agent.

Here are the available specialists:
{specialist_descriptions}

Analyze the following user request and choose the best specialist to handle it. Provide your decision in the required format.

User Request: "{{user_request}}""""
    )
    
    controller_llm = llm.with_structured_output(ControllerDecision, method="function_calling")
    chain = prompt | controller_llm
    
    decision = chain.invoke({"user_request": state['user_request']})
    console.print(f"[yellow]Routing decision:[/yellow] Send to [bold]{decision.next_agent}[/bold]. [italic]Reason: {decision.reasoning}[/italic]")
    
    return {"next_agent_to_call": decision.next_agent}

print("Meta-Controller node defined successfully.")

SyntaxError: unterminated string literal (detected at line 27) (243753288.py, line 27)

## 阶段 3：组装并运行图

使用 LangGraph 连接一切：图从元控制器开始，根据决策用条件边将状态路由到对应专家节点；专家运行结束后图结束。


In [ ]:
# Build the graph
workflow = StateGraph(MetaAgentState)

# Add nodes for the controller and each specialist
workflow.add_node("meta_controller", meta_controller_node)
workflow.add_node("Generalist", generalist_node)
workflow.add_node("Researcher", research_agent_node)
workflow.add_node("Coder", coding_agent_node)

# Set the entry point
workflow.set_entry_point("meta_controller")

# Define the conditional routing logic
def route_to_specialist(state: MetaAgentState) -> str:
    """Reads the controller's decision and returns the name of the node to route to."""
    return state["next_agent_to_call"]

workflow.add_conditional_edges(
    "meta_controller",
    route_to_specialist,
    {
        "Generalist": "Generalist",
        "Researcher": "Researcher",
        "Coder": "Coder"
    }
)

# After any specialist runs, the process ends
workflow.add_edge("Generalist", END)
workflow.add_edge("Researcher", END)
workflow.add_edge("Coder", END)

meta_agent = workflow.compile()
print("Meta-Controller agent graph compiled successfully.")

from IPython.display import Image, display
display(Image(meta_agent.get_graph().draw_mermaid_png()))

Meta-Controller agent graph compiled successfully.


## 阶段 4：演示

用多种提示测试元控制器，检验是否正确分派到对应专家。


In [6]:
def run_agent(query: str):
    result = meta_agent.invoke({"user_request": query})
    console.print("\n[bold]Final Response:[/bold]")
    console.print(Markdown(result['generation']))

# Test 1: Should be routed to the Generalist
console.print("--- 💬 Test 1: General Conversation ---")
run_agent("Hello, how are you today?")

# Test 2: Should be routed to the Researcher
console.print("\n--- 🔬 Test 2: Research Question ---")
run_agent("What were NVIDIA's latest financial results?")

# Test 3: Should be routed to the Coder
console.print("\n--- 💻 Test 3: Coding Request ---")
run_agent("Can you write me a python function to calculate the nth fibonacci number?")

--- 💬 Test 1: General Conversation ---


--- 🧠 Meta-Controller Analyzing Request ---
Routing decision: Send to Generalist. Reason: The user's request is a simple greeting, which falls under the category of casual conversation handled by the Generalist agent.



Final Response:


Hello there! How can I help you today?


--- 🔬 Test 2: Research Question ---


...

NVIDIA's latest financial results, for the quarter ending in April 2024, were exceptionally strong. They reported revenue of $26.04 billion, a significant increase year-over-year, driven largely by their Data Center revenue which hit a record $22.6 billion. Their GAAP earnings per diluted share were $5.98.


--- 💻 Test 3: Coding Request ---


--- 🧠 Meta-Controller Analyzing Request ---
Routing decision: Send to Coder. Reason: The user is explicitly asking for a Python function, which is a coding task. The Coder agent is the specialist for this.



Final Response:


```python
def fibonacci(n):
    """Calculates the nth Fibonacci number."""
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b
```

## 小结

本 Notebook 成功实现了 **元控制器** 架构。测试表明其核心作用：作为智能、动态的路由器。

1. 简单问候被正确识别并交给 **通才**。
2. 近期财报类查询被分派给 **研究智能体**，由其搜索工具获取最新信息。
3. 代码片段请求被路由到 **编程智能体**，返回格式良好且正确的函数。

该模式极适合构建可扩展、可维护的 AI 系统。职责分离后，各专家可独立改进而不相互牵连；整体智能可通过新增更强专家并让元控制器知晓而增强。尽管控制器可能成为瓶颈，其作为灵活编排者的角色仍是高级智能体设计的基石。
